# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [2]:
#@title 1.1 — Install
# Chạy dòng dưới nếu cần cài đặt thư viện vào môi trường hiện tại:
# %pip install -q neo4j sentence-transformers faiss-cpu groq openai pandas numpy pyarrow tqdm networkx datasets python-dotenv
print("✅ Hãy đảm bảo đã kích hoạt virtualenv và cài đặt: pip install -r requirements.txt")


✅ Hãy đảm bảo đã kích hoạt virtualenv và cài đặt: pip install -r requirements.txt


In [21]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

# Tự động nạp biến môi trường từ file .env (hỗ trợ VS Code / Local)
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None and value != "":
            return value
    except Exception:
        pass
    val = os.environ.get(name)
    if val is not None and val != "":
        return val
    return default

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "llama-3.1-8b-instant")
GROQ_FALLBACK_MODEL = "llama-3.1-8b-instant"

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
AI_API_KEY = get_secret("AI_API_KEY", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

# Tự động tạo thư mục outputs và thiết lập đường dẫn cục bộ cho VS Code
os.makedirs("outputs", exist_ok=True)
DATA_PATH = os.environ.get("DATA_PATH", "outputs/hackernoon_subset.csv")

LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [4]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from pathlib import Path

from datasets import load_dataset
from tqdm.auto import tqdm

# Nạp .env khi cell được chạy độc lập trong VS Code/local.
try:
    from dotenv import load_dotenv
    workspace_dir = Path.cwd()
    env_candidates = [workspace_dir / ".env", Path(__file__).resolve().parent / ".env"] if "__file__" in globals() else [workspace_dir / ".env"]
    for env_path in env_candidates:
        if env_path.exists():
            load_dotenv(env_path, override=False)
            print(f"Đã nạp cấu hình từ: {env_path}")
            break
except Exception:
    pass

# Ưu tiên Colab Secrets, sau đó đến biến môi trường đã được nạp từ .env.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = DATA_PATH
os.makedirs(os.path.dirname(OUTPUT_CSV) or ".", exist_ok=True)

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 1_000_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy đặt HF_TOKEN trong file .env hoặc Colab Secrets."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN trong .env/Colab Secrets, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng Internet."
    )
    raise


Đã nạp cấu hình từ: d:\AITHUCCHIEN\LAB\DAY19\K4-Track3-2A202601614-TongTienManh\.env
Đang kết nối luồng dữ liệu (streaming)...
Đang ghi dữ liệu vào: outputs/hackernoon_subset.csv


Đang tải (MB): 100%|██████████| 300.0/300 [09:37<00:00,  1.93s/MB]                



[DỪNG] Đã đạt giới hạn dung lượng: 300.00 MB (Tổng: 514,417 dòng)
✅ Hoàn thành: d:\AITHUCCHIEN\LAB\DAY19\K4-Track3-2A202601614-TongTienManh\outputs\hackernoon_subset.csv
   Rows: 514,417
   Size: 300.00 MB


In [28]:
#@title 1.4 — Neo4j connection + schema (auto-reconnect, fallback scheme)
from urllib.parse import urlparse

from neo4j import GraphDatabase
from neo4j.exceptions import (
    AuthError,
    ConfigurationError,
    ServiceUnavailable,
    SessionExpired,
    TransientError,
)

driver = None
NEO4J_ACTIVE_URI = ""

# Cấu hình chống "Unable to retrieve routing information":
# - max_connection_lifetime ngắn để không tái dùng socket đã bị Aura/idle-proxy đóng.
# - keep_alive giữ TCP sống khi notebook idle giữa các cell.
DRIVER_CONFIG = dict(
    max_connection_lifetime=600,
    connection_acquisition_timeout=60,
    connection_timeout=30,
    max_transaction_retry_time=30,
    keep_alive=True,
)

RUN_CYPHER_MAX_RETRIES = 4

def _uri_candidates(uri):
    """
    Sinh danh sách URI để thử lần lượt.
    `neo4j+s` cần lấy routing table; nếu bước đó fail (proxy/firewall/DNS SRV),
    `bolt+s` kết nối trực tiếp tới instance nên thường vẫn chạy được.
    `+ssc` bỏ qua verify chain certificate (một số mạng dùng TLS inspection).
    """
    parsed = urlparse(uri if "://" in uri else f"neo4j+s://{uri}")
    host = parsed.netloc or parsed.path
    scheme = (parsed.scheme or "neo4j+s").lower()
    variants = {
        "neo4j+s": ["neo4j+s", "bolt+s", "neo4j+ssc", "bolt+ssc"],
        "neo4j+ssc": ["neo4j+ssc", "bolt+ssc"],
        "bolt+s": ["bolt+s", "neo4j+s"],
        "bolt+ssc": ["bolt+ssc", "neo4j+ssc"],
        "neo4j": ["neo4j", "bolt"],
        "bolt": ["bolt", "neo4j"],
    }.get(scheme, [scheme])
    return [f"{s}://{host}" for s in variants]

def _close_driver():
    global driver
    try:
        if driver is not None:
            driver.close()
    except Exception:
        pass
    driver = None

def connect_neo4j(quiet=False):
    global driver, NEO4J_ACTIVE_URI
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets (NEO4J_URI / NEO4J_PASSWORD).")

    _close_driver()
    errors = []
    for uri in _uri_candidates(NEO4J_URI):
        candidate = None
        try:
            candidate = GraphDatabase.driver(
                uri, auth=(NEO4J_USER, NEO4J_PASSWORD), **DRIVER_CONFIG
            )
            candidate.verify_connectivity()
            # verify_connectivity() có thể pass nhưng database đích vẫn không mở được -> probe thật.
            with candidate.session(database=NEO4J_DATABASE) as session:
                session.run("RETURN 1 AS ok").consume()
            driver = candidate
            NEO4J_ACTIVE_URI = uri
            if not quiet:
                print(f"✅ Neo4j connected: {uri} (database={NEO4J_DATABASE})")
            return driver
        except (AuthError, ConfigurationError) as e:
            if candidate is not None:
                try:
                    candidate.close()
                except Exception:
                    pass
            raise RuntimeError(
                f"Neo4j từ chối đăng nhập/sai cấu hình ({type(e).__name__}): {e}. "
                "Kiểm tra NEO4J_USER / NEO4J_PASSWORD / NEO4J_DATABASE trong .env."
            ) from e
        except Exception as e:
            if candidate is not None:
                try:
                    candidate.close()
                except Exception:
                    pass
            first_line = str(e).splitlines()[0] if str(e) else ""
            errors.append(f"  - {uri} -> {type(e).__name__}: {first_line[:160]}")

    raise ServiceUnavailable(
        "Không kết nối được Neo4j với mọi scheme đã thử:\n"
        + "\n".join(errors)
        + "\nKiểm tra: (1) AuraDB instance đang ở trạng thái Running (free tier tự pause sau 3 ngày không dùng), "
        "(2) NEO4J_URI đúng instance id, (3) mạng/firewall cho phép port 7687."
    )

def ensure_neo4j(verbose=False):
    """Bảo đảm có driver còn sống; tự reconnect nếu connection đã chết vì idle/pause."""
    global driver
    if driver is None:
        return connect_neo4j(quiet=not verbose)
    try:
        driver.verify_connectivity()
        if verbose:
            print(f"✅ Neo4j OK: {NEO4J_ACTIVE_URI} (database={NEO4J_DATABASE})")
        return driver
    except Exception as e:
        print(f"⚠️ Connection Neo4j đã chết ({type(e).__name__}). Đang reconnect...")
        return connect_neo4j(quiet=not verbose)

def run_cypher(query, **params):
    """
    Chạy Cypher với retry + reconnect.
    Retry các lỗi hạ tầng (ServiceUnavailable/SessionExpired/TransientError/OSError);
    lỗi cú pháp Cypher hoặc constraint (ClientError) fail ngay để không che bug.
    """
    global driver
    last_err = None
    for attempt in range(RUN_CYPHER_MAX_RETRIES):
        try:
            if driver is None:
                connect_neo4j(quiet=True)
            with driver.session(database=NEO4J_DATABASE) as session:
                result = session.run(query, **params)
                rows = [r.data() for r in result]
                result.consume()
            return rows
        except (ServiceUnavailable, SessionExpired, TransientError, OSError) as e:
            last_err = e
            if attempt == RUN_CYPHER_MAX_RETRIES - 1:
                break
            wait = min(15, 2 ** attempt) + random.random()
            print(
                f"⚠️ Neo4j {type(e).__name__} (lần {attempt + 1}/{RUN_CYPHER_MAX_RETRIES - 1}). "
                f"Reconnect sau {wait:.1f}s..."
            )
            _close_driver()
            time.sleep(wait)
            try:
                connect_neo4j(quiet=True)
            except Exception as reconnect_err:
                last_err = reconnect_err

    raise RuntimeError(
        "Neo4j không truy cập được sau nhiều lần thử. Kiểm tra: "
        "(1) AuraDB instance đang Running (free tier tự pause sau 3 ngày idle — vào console.neo4j.io bấm Resume), "
        "(2) NEO4J_URI/USER/PASSWORD trong .env, (3) mạng/firewall cho port 7687. "
        f"Lỗi cuối: {type(last_err).__name__}: {last_err}"
    ) from last_err

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()

✅ Neo4j connected: neo4j+s://28944c2f.databases.neo4j.io (database=neo4j)
✅ Schema ready.


In [6]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    candidates = [
        Path(path),
        Path("outputs/hackernoon_subset.csv"),
        Path("hackernoon_subset.csv"),
        Path("/content/hackernoon_subset.csv"),
    ]
    actual_path = None
    for cand in candidates:
        if cand.exists():
            actual_path = cand
            break
    if actual_path is None:
        raise FileNotFoundError(f"Không tìm thấy file dữ liệu tại {path}. Hãy chạy cell 1.3 để tải hoặc đặt file vào outputs/hackernoon_subset.csv")
    
    if actual_path.suffix.lower() == ".csv":
        return pd.read_csv(actual_path)
    if actual_path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(actual_path, lines=True)
    if actual_path.suffix.lower() == ".json":
        return pd.read_json(actual_path)
    if actual_path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(actual_path)
    raise ValueError(f"Unsupported: {actual_path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "article_text", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}") for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())


Exact dedup: 245,324 -> 212,212


Chunking: 100%|██████████| 1500/1500 [00:00<00:00, 17376.26it/s]


,chunk_id,article_id,title,published_date,text
0,d38fc817b7dc3eeeb535cc492f3fc9112cfa2011::c0000,d38fc817b7dc3eeeb535cc492f3fc9112cfa2011,Information Services Corporation Non-GAAP EPS of C$0.51 revenue of C$53.3M,2023-08-03,To ensure this doesn’t happen in the future please enable Javascript and cookies in your browser. Is this happening ...
1,830dbc6ea3082bb64be00b5075dff4fb38eb33c5::c0000,830dbc6ea3082bb64be00b5075dff4fb38eb33c5,How GSA’s Technology Transformation Services is Harnessing Change in Tech Modernization,2023-05-24,One component of GSA in particular Technology Transformation Services carries much of this mission by using modern m...
2,8cbfe069b03305566d73891bedbd87e5d5a9c055::c0000,8cbfe069b03305566d73891bedbd87e5d5a9c055,Information Technology,2023-05-18,At the most recent Berkshire Hathaway Inc. (NYSE: BRK-B) investors conference in early May Warren Buffett offered so...
3,331537d8f978a369b442b3fa86421f30592878b4::c0000,331537d8f978a369b442b3fa86421f30592878b4,Ryan Specialty Signs Definitive Agreement To Acquire Socius Insurance,2023-05-23,Ryan Specialty (NYSE:RYAN) a leading international specialty insurance firm is pleased to announce that it has signe...
4,55a09cbc43c41ffb2dd90b77deda18e40747ea91::c0000,55a09cbc43c41ffb2dd90b77deda18e40747ea91,Transact Campus Partnership Lands Talkiatry Services on Campus Transact Apps,2023-09-05,The partnership will “provide students with access to quality psychiatric services and offers an accessible and affo...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [7]:
#@title 1.6 — LLM wrapper có retry + JSON parsing & Near-Dedup (MinHash/LSH & Embedding+ANN)
# ==============================================================================
# PHẦN A: NEAR-DEDUPLICATION (MinHash/LSH & FAISS Embedding+ANN)
# Thiết kế O(N) / O(N log N) - Tuyệt đối không dùng pairwise O(N^2) toàn dataset
# ==============================================================================

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

class MinHashLSH:
    """
    MinHash + Locality-Sensitive Hashing (LSH) cho bài toán Near-Deduplication.
    Độ phức tạp: O(N) indexing + O(N) bucketing (tránh hoàn toàn O(N^2) pairwise comparisons).
    """
    def __init__(self, num_perm=128, num_bands=32, seed=42):
        assert num_perm % num_bands == 0, "num_perm phải chia hết cho num_bands"
        self.num_perm = num_perm
        self.num_bands = num_bands
        self.rows_per_band = num_perm // num_bands
        self.seed = seed
        self.prime = (1 << 61) - 1  # Mersenne prime 2^61 - 1
        rng = np.random.RandomState(seed)
        self.a = rng.randint(1, self.prime - 1, size=num_perm, dtype=np.int64)
        self.b = rng.randint(0, self.prime - 1, size=num_perm, dtype=np.int64)
        self.buckets = defaultdict(list)

    @staticmethod
    def get_shingles(text, k=2):
        """Tạo word-level k-shingles (k=2) từ văn bản."""
        words = norm_space(text).lower().split()
        if len(words) < k:
            return set(words) if words else {""}
        return {" ".join(words[i:i+k]) for i in range(len(words) - k + 1)}

    def compute_signature(self, shingles):
        """Tính vector chữ ký MinHash độ dài num_perm (vectorized)."""
        if not shingles:
            return np.zeros(self.num_perm, dtype=np.int64)

        shingle_hashes = np.array([
            int(hashlib.sha1(s.encode("utf-8")).hexdigest()[:15], 16)
            for s in shingles
        ], dtype=np.int64)

        # (a * x + b) % prime
        x = shingle_hashes.reshape(1, -1)
        a = self.a.reshape(-1, 1)
        b = self.b.reshape(-1, 1)
        hash_values = (a * x + b) % self.prime
        sig = np.min(hash_values, axis=1)
        return sig

    def insert(self, doc_id, signature):
        """Nạp chữ ký vào các band buckets của LSH."""
        for band_idx in range(self.num_bands):
            start = band_idx * self.rows_per_band
            band_sig = tuple(signature[start:start + self.rows_per_band])
            bucket_key = (band_idx, hash(band_sig))
            self.buckets[bucket_key].append(doc_id)

    def query_candidate_pairs(self):
        """Trích xuất các cặp tài liệu ứng viên cùng rơi vào ít nhất 1 bucket."""
        candidate_pairs = set()
        for bucket, doc_ids in self.buckets.items():
            if len(doc_ids) > 1:
                for i in range(len(doc_ids)):
                    for j in range(i + 1, len(doc_ids)):
                        u, v = doc_ids[i], doc_ids[j]
                        if u != v:
                            candidate_pairs.add((min(u, v), max(u, v)))
        return candidate_pairs

def jaccard_similarity(set_a, set_b):
    """Tính độ tương đồng Jaccard chính xác giữa 2 tập shingle."""
    if not set_a or not set_b:
        return 0.0
    intersection = len(set_a.intersection(set_b))
    union = len(set_a.union(set_b))
    return intersection / union if union > 0 else 0.0

def near_dedup_guard(title_a, title_b, text_a, text_b, min_words=20):
    """
    Lexical & Structural Guard chống False Positives:
    - Loại bỏ văn bản quá ngắn (< min_words) dễ gây xung đột ngẫu nhiên.
    - So khớp tiêu đề & độ dài để chặn gộp các bản tin mẫu (template earnings / boilerplate).
    """
    words_a, words_b = len(text_a.split()), len(text_b.split())
    if words_a < min_words or words_b < min_words:
        return False, "REJECT_TOO_SHORT"

    if title_a and title_b:
        t_ratio = SequenceMatcher(None, title_a.lower(), title_b.lower()).ratio()
        len_ratio = min(words_a, words_b) / max(words_a, words_b)
        if t_ratio < 0.20 and len_ratio < 0.4:
            return False, "REJECT_TITLE_AND_LEN_MISMATCH"

    return True, "OK"

def near_dedup_minhash_lsh(news_df, threshold=0.80, num_perm=128, num_bands=32, min_words=20, shingle_k=2):
    """
    Pipeline Near-Deduplication chuẩn Production bằng MinHash + LSH.
    Returns:
      - deduped_df: DataFrame đã khử trùng lặp (giữ canonical doc).
      - audit_df: Bảng audit toàn bộ candidate pairs kèm điểm tương đồng và quyết định merge.
    """
    if len(news_df) <= 1:
        return news_df.copy(), pd.DataFrame()

    print(f"Running MinHash/LSH Near-Dedup on {len(news_df):,} articles (threshold={threshold}, bands={num_bands})...")
    lsh = MinHashLSH(num_perm=num_perm, num_bands=num_bands, seed=SEED)

    doc_meta = {}
    doc_shingles = {}

    for idx, r in enumerate(news_df.itertuples(index=False)):
        doc_id = getattr(r, "article_id", str(idx))
        text = str(getattr(r, "text", ""))
        title = str(getattr(r, "title", ""))
        shingles = MinHashLSH.get_shingles(text if len(text.split()) >= min_words else f"{title} {text}", k=shingle_k)
        doc_shingles[doc_id] = shingles
        doc_meta[doc_id] = {
            "index": idx,
            "title": title,
            "text": text,
            "published_date": getattr(r, "published_date", ""),
            "length": len(text)
        }
        sig = lsh.compute_signature(shingles)
        lsh.insert(doc_id, sig)

    candidate_pairs = lsh.query_candidate_pairs()
    print(f"Extracted {len(candidate_pairs):,} candidate pairs via LSH buckets (avoided {len(news_df)*(len(news_df)-1)//2:,} all-pairs).")

    all_doc_ids = list(doc_meta.keys())
    id_to_idx = {did: i for i, did in enumerate(all_doc_ids)}
    uf = UF(len(all_doc_ids))

    audit_rows = []

    for id_a, id_b in candidate_pairs:
        shingles_a = doc_shingles[id_a]
        shingles_b = doc_shingles[id_b]
        jaccard = jaccard_similarity(shingles_a, shingles_b)

        meta_a = doc_meta[id_a]
        meta_b = doc_meta[id_b]

        if jaccard >= threshold:
            guard_ok, guard_reason = near_dedup_guard(meta_a["title"], meta_b["title"], meta_a["text"], meta_b["text"], min_words)
            if guard_ok:
                decision = "MERGE_MINHASH_LSH"
                uf.union(id_to_idx[id_a], id_to_idx[id_b])
            else:
                decision = f"REJECT_GUARD_{guard_reason}"
        else:
            decision = "REJECT_BELOW_THRESHOLD"

        audit_rows.append({
            "left_id": id_a,
            "right_id": id_b,
            "left_title": meta_a["title"][:60],
            "right_title": meta_b["title"][:60],
            "similarity": round(float(jaccard), 4),
            "decision": decision
        })

    clusters = defaultdict(list)
    for did, idx in id_to_idx.items():
        root = uf.find(idx)
        clusters[root].append(did)

    keep_indices = []
    for root, member_ids in clusters.items():
        # Chọn canonical document: ưu tiên bài có ngày sớm nhất, rồi đến bài dài nhất
        best_did = sorted(
            member_ids,
            key=lambda did: (
                doc_meta[did]["published_date"] or "9999-99-99",
                -doc_meta[did]["length"]
            )
        )[0]
        keep_indices.append(doc_meta[best_did]["index"])

    keep_indices = sorted(keep_indices)
    deduped_df = news_df.iloc[keep_indices].reset_index(drop=True)
    audit_df = pd.DataFrame(audit_rows)
    if not audit_df.empty:
        audit_df = audit_df.sort_values(by="similarity", ascending=False).reset_index(drop=True)

    print(f"MinHash/LSH Result: {len(news_df):,} -> {len(deduped_df):,} articles (removed {len(news_df) - len(deduped_df):,} duplicates).")
    return deduped_df, audit_df

def near_dedup_ann(news_df, threshold=0.88, top_k=5, min_words=20):
    """
    Phương án bổ trợ: Near-Dedup dùng Sentence-Transformers + FAISS IndexFlatIP (ANN Sub-linear / Top-K).
    Tránh O(N^2) pairwise bằng cách chỉ truy vấn Top-K nearest neighbors.
    """
    if len(news_df) <= 1:
        return news_df.copy(), pd.DataFrame()

    print(f"Running Embedding+ANN Near-Dedup on {len(news_df):,} articles (threshold={threshold}, top_k={top_k})...")
    embedder = get_embedder()
    texts = [f"{r.title}: {r.text[:200]}" for r in news_df.itertuples(index=False)]
    vecs = embedder.encode(texts, batch_size=128, show_progress_bar=False, normalize_embeddings=True).astype("float32")

    dim = vecs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vecs)
    sims, nbrs = index.search(vecs, min(top_k + 1, len(texts)))

    uf = UF(len(news_df))
    audit_rows = []
    seen_pairs = set()

    for i in range(len(news_df)):
        for score, j in zip(sims[i], nbrs[i]):
            if j < 0 or i >= j:
                continue
            pair = (i, j)
            if pair in seen_pairs:
                continue
            seen_pairs.add(pair)

            sim_val = float(score)
            row_a = news_df.iloc[i]
            row_b = news_df.iloc[j]

            if sim_val >= threshold:
                guard_ok, guard_reason = near_dedup_guard(row_a.title, row_b.title, row_a.text, row_b.text, min_words)
                if guard_ok:
                    decision = "MERGE_ANN"
                    uf.union(i, j)
                else:
                    decision = f"REJECT_GUARD_{guard_reason}"
            else:
                decision = "REJECT_BELOW_THRESHOLD"

            audit_rows.append({
                "left_id": str(getattr(row_a, "article_id", i)),
                "right_id": str(getattr(row_b, "article_id", j)),
                "left_title": str(getattr(row_a, "title", ""))[:60],
                "right_title": str(getattr(row_b, "title", ""))[:60],
                "similarity": round(sim_val, 4),
                "decision": decision
            })

    clusters = defaultdict(list)
    for i in range(len(news_df)):
        clusters[uf.find(i)].append(i)

    keep_indices = [sorted(members, key=lambda idx: -len(news_df.iloc[idx].text))[0] for members in clusters.values()]
    keep_indices = sorted(keep_indices)
    deduped_df = news_df.iloc[keep_indices].reset_index(drop=True)
    audit_df = pd.DataFrame(audit_rows)
    if not audit_df.empty:
        audit_df = audit_df.sort_values(by="similarity", ascending=False).reset_index(drop=True)
    return deduped_df, audit_df

# ==============================================================================
# PHẦN B: LLM WRAPPER CÓ RETRY + JSON PARSING
# ==============================================================================
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage


In [22]:
#@title 1.6b — Groq model fallback theo model được cấp quyền

def _is_model_not_found(error):
    status_code = getattr(error, "status_code", None)
    message = str(error).lower()
    return status_code == 404 or "model_not_found" in message or "does not exist" in message


def _available_groq_models():
    try:
        response = groq_client.models.list()
        return [getattr(model, "id", "") for model in response.data if getattr(model, "id", "")]
    except Exception:
        return []


def _choose_groq_model(requested_model):
    available = _available_groq_models()
    if not available:
        return None, []

    preferred = [
        "openai/gpt-oss-120b",
        "openai/gpt-oss-20b",
        "qwen/qwen3.6-27b",
        "groq/compound",
        "groq/compound-mini",
    ]
    selected = next((model for model in preferred if model in available), None)
    if selected is None:
        selected = next(
            (model for model in available if any(token in model.lower() for token in ("llama", "qwen", "gpt"))),
            available[0],
        )
    return selected, available


def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    global GROQ_MODEL
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")

    requested_model = model or GROQ_MODEL
    if not requested_model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    models_to_try = [requested_model]
    fallback_model = globals().get("GROQ_FALLBACK_MODEL", "")
    if fallback_model and fallback_model not in models_to_try:
        models_to_try.append(fallback_model)

    last = None
    available = []
    for selected_model in models_to_try:
        for attempt in range(max_retries):
            try:
                kwargs = {
                    "model": selected_model,
                    "messages": messages,
                    "temperature": 0.0,
                }
                if json_mode:
                    kwargs["response_format"] = {"type": "json_object"}

                resp = groq_client.chat.completions.create(**kwargs)
                usage = {}
                if getattr(resp, "usage", None):
                    usage = {
                        "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                        "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                        "total_tokens": getattr(resp.usage, "total_tokens", None),
                    }
                if selected_model != requested_model:
                    GROQ_MODEL = selected_model
                    print(f"⚠️ Model {requested_model!r} không khả dụng; đã chuyển sang {selected_model!r}.")
                return resp.choices[0].message.content, usage
            except Exception as error:
                last = error
                if _is_model_not_found(error):
                    break
                if attempt < max_retries - 1:
                    time.sleep(min(20, 2**attempt + random.random()))

        if _is_model_not_found(last):
            discovered_model, available = _choose_groq_model(requested_model)
            if discovered_model and discovered_model not in models_to_try:
                models_to_try.append(discovered_model)
            continue

    available_text = ", ".join(available[:20]) if available else "không lấy được danh sách model"
    raise RuntimeError(
        f"Groq không cấp quyền cho model {requested_model!r}. "
        f"Các model API key nhìn thấy: {available_text}. "
        "Hãy đặt GROQ_MODEL trong .env bằng một model trong danh sách trên."
    ) from last


def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [9]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

Coref:   1%|▏         | 1/80 [00:04<06:10,  4.69s/it]

⚠️ Model 'llama-3.3-70b-versatile' không khả dụng; đã chuyển sang 'openai/gpt-oss-120b'.


Coref: 100%|██████████| 80/80 [15:14<00:00, 11.43s/it]


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [10]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    return pd.DataFrame(triples), pd.DataFrame(errors)

raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
display(raw_triples_df.head())

NER+RE: 100%|██████████| 100/100 [19:23<00:00, 11.63s/it]


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,NSPR,Company,DEVELOPED,CGuard™ Embolic Prevention Stent System (EPS),Technology,7335832cb8ae46a8eec160bf9e3031938f22a735::c0000,2023-10-03,(Nasdaq: NSPR) developer of the CGuard™ Embolic Prevention Stent System (EPS) for the prevention of stroke,1.00
1,Microsoft,Company,ACQUIRED,Activision Blizzard,Company,67cfec0c6040e230e1b2ecf2e1038e4a543b45c2::c0000,2023-03-02,The European Union regulator the European Commission is likely to approve Microsoft''s Activision Blizzard acquisition,1.00
2,Berkley Re Solutions,Company,PARTNERED_WITH,Barton Mutual Insurance Company,Company,0683886718484f0cc15279dce2be548cc25f2fd8::c0000,2023-02-07,Berkley Re Solutions today announced a partnership with Barton Mutual Insurance Company to introduce a first-of-its-...,0.95


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [11]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {
    "inc", "incorporated", "corp", "corporation", "ltd", "limited",
    "llc", "plc", "co", "company", "gmbh", "sa", "ag", "nv", "bv",
    "group", "holdings", "holding", "technologies", "technology", "tech",
    "systems", "solutions", "services", "labs", "lab", "ai"
}

KNOWN_TICKERS = {
    "msft": "Microsoft",
    "aapl": "Apple",
    "goog": "Google",
    "googl": "Google",
    "amzn": "Amazon",
    "meta": "Meta",
    "nvda": "Nvidia",
    "tsla": "Tesla",
    "ibm": "IBM",
    "intc": "Intel",
    "orcl": "Oracle",
    "nflx": "Netflix",
    "amd": "AMD",
    "crm": "Salesforce",
    "uber": "Uber",
    "abnb": "Airbnb",
    "pltr": "Palantir",
    "csco": "Cisco",
    "adbe": "Adobe",
    "qcom": "Qualcomm",
    "bidu": "Baidu",
    "baba": "Alibaba",
    "spot": "Spotify",
    "sq": "Block",
    "pypl": "PayPal",
    "twtr": "Twitter",
    "snap": "Snap",
}

MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "google inc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
    "apple corp": "Apple",
    "nvda": "Nvidia",
    "nvidia corp": "Nvidia",
    "amzn": "Amazon",
    "amazon com": "Amazon",
    "amazon com inc": "Amazon",
    "tsla": "Tesla",
    "tesla inc": "Tesla",
    "tesla motors": "Tesla",
}
for tk, comp in KNOWN_TICKERS.items():
    MANUAL_ALIASES.setdefault(tk, comp)

PRODUCT_MODIFIERS = {
    "music", "cloud", "watch", "phone", "search", "maps", "pay", "tv",
    "quest", "teams", "azure", "office", "studio", "hub", "api", "browser",
    "os", "suite", "drive", "mail", "photos", "play", "prime", "alexa",
    "gemini", "chatgpt", "copilot", "vision", "pro", "max", "mini", "plus",
    "air", "lens", "wallet", "card", "health", "workspace", "connect",
    "ads", "analytics", "database", "assistant", "news", "fit", "store",
    "hardware", "software", "auto", "car"
}

NICKNAME_MAP = {
    "sam": {"samuel", "sam"},
    "samuel": {"sam", "samuel"},
    "bill": {"william", "bill", "billy"},
    "william": {"bill", "william", "billy"},
    "steve": {"stephen", "steven", "steve"},
    "steven": {"steve", "steven", "stephen"},
    "stephen": {"steve", "steven", "stephen"},
    "bob": {"robert", "bob", "bobby"},
    "robert": {"bob", "robert", "bobby"},
    "mike": {"michael", "mike"},
    "michael": {"mike", "michael"},
    "dave": {"david", "dave"},
    "david": {"dave", "david"},
    "dan": {"daniel", "dan", "danny"},
    "daniel": {"dan", "daniel", "danny"},
    "alex": {"alexander", "alex", "alexandra"},
    "alexander": {"alex", "alexander"},
    "tim": {"timothy", "tim"},
    "timothy": {"tim", "timothy"},
    "larry": {"lawrence", "larry"},
    "lawrence": {"larry", "lawrence"},
    "jim": {"james", "jim", "jimmy"},
    "james": {"jim", "james", "jimmy"},
    "andy": {"andrew", "andy"},
    "andrew": {"andy", "andrew"},
    "matt": {"matthew", "matt"},
    "matthew": {"matt", "matthew"},
    "chris": {"christopher", "chris"},
    "christopher": {"chris", "christopher"},
    "jeff": {"jeffrey", "jeff"},
    "jeffrey": {"jeff", "jeffrey"},
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", str(name or "")).lower().strip()
    s = re.sub(r"^(nasdaq|nyse|ticker)\s*:\s*", "", s)
    s = re.sub(r"^\$", "", s)
    s = re.sub(r"[^\w\s\-]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def is_ticker_str(s):
    raw = str(s or "").strip()
    norm = norm_entity(raw)
    if norm in KNOWN_TICKERS:
        return True
    if raw.isupper() and 2 <= len(raw) <= 5 and raw.isalpha():
        return True
    return False

def is_first_name_match(fn1, fn2):
    if fn1 == fn2:
        return True
    if fn1 in NICKNAME_MAP.get(fn2, set()) or fn2 in NICKNAME_MAP.get(fn1, set()):
        return True
    if (len(fn1) == 1 and fn2.startswith(fn1)) or (len(fn2) == 1 and fn1.startswith(fn2)):
        return True
    return False

def merge_guard(a, b, typ=None):
    na, nb = norm_entity(a), norm_entity(b)
    if not na or not nb:
        return False, "REJECT_EMPTY"
    if na == nb:
        return True, "OK_EXACT"

    is_tk_a = is_ticker_str(a)
    is_tk_b = is_ticker_str(b)
    if is_tk_a or is_tk_b:
        canon_a = MANUAL_ALIASES.get(na)
        canon_b = MANUAL_ALIASES.get(nb)
        if canon_a and canon_b and norm_entity(canon_a) == norm_entity(canon_b):
            return True, "OK_TICKER_ALIAS_MATCH"
        if is_tk_a and canon_a and norm_entity(canon_a) == strip_suffix(nb):
            return True, "OK_TICKER_TO_COMPANY"
        if is_tk_b and canon_b and norm_entity(canon_b) == strip_suffix(na):
            return True, "OK_TICKER_TO_COMPANY"
        if is_tk_a != is_tk_b:
            return False, "REJECT_UNMAPPED_TICKER"

    if typ == "Person":
        toks_a = na.split()
        toks_b = nb.split()
        if len(toks_a) >= 2 and len(toks_b) >= 2:
            fn_a, ln_a = toks_a[0], toks_a[-1]
            fn_b, ln_b = toks_b[0], toks_b[-1]
            if ln_a != ln_b: return False, "REJECT_PERSON_LAST_NAME_MISMATCH"
            if not is_first_name_match(fn_a, fn_b): return False, "REJECT_PERSON_FIRST_NAME_MISMATCH"
            return True, "OK_PERSON_NAME_MATCH"
        elif len(toks_a) == 1 or len(toks_b) == 1:
            single = toks_a[0] if len(toks_a) == 1 else toks_b[0]
            multi = toks_b if len(toks_a) == 1 else toks_a
            if single == multi[-1]: return False, "REJECT_PERSON_SINGLE_LASTNAME_AMBIGUOUS"
            if single == multi[0]: return False, "REJECT_PERSON_SINGLE_FIRSTNAME_AMBIGUOUS"

    sa, sb = strip_suffix(na), strip_suffix(nb)
    if sa == sb and sa: return True, "OK_SUFFIX_MATCH"

    toks_sa = set(sa.split())
    toks_sb = set(sb.split())
    if toks_sa and toks_sb:
        if (toks_sa.issubset(toks_sb) or toks_sb.issubset(toks_sa)) and abs(len(toks_sa)-len(toks_sb)) >= 1:
            extra = (toks_sb - toks_sa) if len(toks_sb) > len(toks_sa) else (toks_sa - toks_sb)
            if any(t in PRODUCT_MODIFIERS for t in extra) or len(extra) >= 1:
                return False, "REJECT_PRODUCT_COMPANY_MISMATCH"

    ratio = SequenceMatcher(None, sa, sb).ratio()
    if ratio >= 0.85:
        return True, "OK_FUZZY_RATIO"
    return False, "REJECT_LOW_SIMILARITY"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None
def get_embedder():
    global embedder
    if embedder is None: embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x: self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b: self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    if raw_triples_df.empty:
        return {}, pd.DataFrame()
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]
    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions: display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []
    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({"type": t, "left": display_name[key], "right": MANUAL_ALIASES[norm], "similarity": 1.0, "decision": "MERGE_MANUAL"})

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys: continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(names, batch_size=128, show_progress_bar=False, normalize_embeddings=True).astype("float32")
        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))
        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold: continue
                ok, reason = merge_guard(names[i], names[j], typ=typ)
                audit.append({"type": typ, "left": names[i], "right": names[j], "similarity": float(score), "decision": "MERGE_VECTOR" if ok else f"REJECT_GUARD_{reason}"})
                if ok: uf.union(i, j)
        groups = defaultdict(list)
        for i in range(len(names)): groups[uf.find(i)].append(i)
        for idxs in groups.values():
            best = sorted(idxs, key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower()))[0]
            canonical = names[best]
            for i in idxs: mapping[keys[i]] = canonical
    for key in counts: mapping.setdefault(key, display_name[key])
    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    if raw_df.empty:
        return pd.DataFrame(columns=["source_name", "target_name", "source_name_norm", "target_name_norm", "source_id", "target_id"])
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))
    df["source_name"] = [canon(n,t) for n,t in zip(df["source_raw"], df["source_type"])]
    df["target_name"] = [canon(n,t) for n,t in zip(df["target_raw"], df["target_type"])]
    df["source_name_norm"] = df["source_name"].map(norm_entity)
    df["target_name_norm"] = df["target_name"].map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df["source_type"], df["source_name_norm"])]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df["target_type"], df["target_name_norm"])]
    return df[df["source_id"] != df["target_id"]].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
display(entity_resolution_audit_df.head(20))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1343.21it/s]


""


In [12]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    if triples_df.empty:
        return pd.DataFrame(columns=["id", "name", "name_norm", "type", "aliases", "aliases_norm"])

    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    if nodes_df.empty:
        print("Nodes DataFrame is empty. Skipping bulk insert.")
        return

    for typ in sorted(ALLOWED_NODE_TYPES):
        # Fix: Use bracket notation to avoid conflict with Python's built-in type()
        part = nodes_df[nodes_df['type'] == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    if triples_df.empty:
        print("Triples DataFrame is empty. Skipping bulk insert.")
        return

    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)

In [13]:
#@title 2.4 — Sanity checks

def graph_checks():
    if driver is None:
        print("⚠️ Neo4j chưa kết nối. Bỏ qua sanity checks; hãy chạy connect_neo4j() sau khi kiểm tra secrets và trạng thái AuraDB.")
        return {
            "nodes": None,
            "edges": None,
            "invalid_provenance_edges": None,
        }, pd.DataFrame()

    try:
        invalid = run_cypher("""
        MATCH ()-[r]->()
        WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
        RETURN count(r) AS n
        """)[0]["n"]

        counts = {
            "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
            "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
            "invalid_provenance_edges": invalid,
        }
        print(counts)
        assert invalid == 0

        top = pd.DataFrame(run_cypher("""
        MATCH (n:Entity)
        OPTIONAL MATCH (n)-[r]-()
        WITH n, count(r) AS degree
        RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
        ORDER BY degree DESC LIMIT 15
        """))
        display(top)
        return counts, top
    except Exception as exc:
        error_text = str(exc)
        is_routing_error = (
            "ServiceUnavailable" in type(exc).__name__
            or "routing information" in error_text.lower()
            or "routing table" in error_text.lower()
        )
        if not is_routing_error:
            raise

        print("⚠️ Neo4j không cung cấp được routing table, nên chưa thể chạy sanity checks.")
        print("Kiểm tra: NEO4J_URI (Aura thường dùng neo4j+s://), NEO4J_DATABASE, trạng thái instance và network/firewall.")
        print(f"Chi tiết driver: {error_text}")
        return {
            "nodes": None,
            "edges": None,
            "invalid_provenance_edges": None,
        }, pd.DataFrame()


graph_counts, top_degree_df = graph_checks()

{'nodes': 95, 'edges': 60, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,7b219357277193c25d37d788,Railergy,Company,5
1,95fdfcdea30320fbcd96b512,A-Mark Precious Metals,Company,4
2,8693b055457df450df9cfcdf,Unacademy,Company,3
3,e6c3db5c28c9a15bfafd2e04,Apple,Company,3
4,34f0add239e52a9320be6c79,Tom Gardner,Person,2
5,d77aadb855f548f928c9540b,Manolo Saiz,Person,2
6,f87c100acdc217a40f2716db,Max Homa,Person,2
7,909fcd9c188c8c2429afa468,Google Cloud,Company,2
8,6852b15b342dbda70169ffb7,NSPR,Company,2
9,ef0d22fe11f82bc6b9a281df,SRJ Sports Investments,Company,2


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [14]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches: 100%|██████████| 12/12 [01:03<00:00,  5.26s/it]

Flat vectors: 1500


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [15]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [16]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [17]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [18]:
#@title 4.1 — 5 câu Golden starter
GOLDEN_PATH = "outputs/golden_dataset.csv"

starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which company acquired the startup founded by the creator of technology X, and when?",
        "reference_answer":"Microsoft acquired Nuance Communications.",
        "reference_evidence":"Multiple articles describing acquisition chronology."
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Synthesize the partnership timeline between Company A and Company B across 2021-2023.",
        "reference_answer":"OpenAI and Microsoft: 2019 initial investment, expanded in 2021 and 2023 multi-billion investment with Azure exclusive computing.",
        "reference_evidence":"Articles spanning multiple dates."
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"What AI product was developed by the organization that received funding from both Google and Amazon?",
        "reference_answer":"Anthropic received investment from Google/Amazon and developed Claude AI.",
        "reference_evidence":"News chunks covering cloud partnership and model releases."
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"Microsoft and Azure OpenAI services: shifted from initial investment to full product integration in Office 365.",
        "reference_evidence":"Chronological chunks from the tech dump."
    },
])

golden_df = starter_golden.copy()
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")


,id,group,question,reference_answer,reference_evidence
0,G01,factoid,Who was the CEO of Hugging Face in 2023?,Clément Delangue,Validate against instructor dump.
1,G02,multi-hop,"Which company acquired the startup founded by the creator of technology X, and when?",Microsoft acquired Nuance Communications.,Multiple articles describing acquisition chronology.
2,G03,cross-doc,Synthesize the partnership timeline between Company A and Company B across 2021-2023.,"OpenAI and Microsoft: 2019 initial investment, expanded in 2021 and 2023 multi-billion investment with Azure exclusi...",Articles spanning multiple dates.
3,G04,multi-hop,What AI product was developed by the organization that received funding from both Google and Amazon?,Anthropic received investment from Google/Amazon and developed Claude AI.,News chunks covering cloud partnership and model releases.
4,G05,cross-doc,Identify one technology connected to the same company in at least two news chunks and summarize how the relationship...,Microsoft and Azure OpenAI services: shifted from initial investment to full product integration in Office 365.,Chronological chunks from the tech dump.


In [19]:
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    # Fallback to GROQ_MODEL if JUDGE_MODEL is empty
    model_to_use = JUDGE_MODEL if JUDGE_MODEL else GROQ_MODEL

    if not model_to_use:
        raise RuntimeError("Thiếu model để đánh giá (JUDGE_MODEL hoặc GROQ_MODEL).")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=model_to_use)[0]

    if JUDGE_PROVIDER == "shopaikey":
        if not AI_API_KEY:
            raise RuntimeError("Thiếu AI_API_KEY cho shopaikey.")
        from openai import OpenAI
        client = OpenAI(
            api_key=AI_API_KEY,
            base_url='https://api.shopaikey.com/v1'
        )
        resp = client.chat.completions.create(
            model=model_to_use,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    # Default fallback to Groq if provider is not specifically set or known
    return groq_json(system, user, model=model_to_use)[0]

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [24]:
from groq import Groq

GROQ_MODEL = "qwen/qwen3.6-27b"
GROQ_FALLBACK_MODEL = "openai/gpt-oss-20b"
JUDGE_PROVIDER = "groq"
JUDGE_MODEL = "openai/gpt-oss-20b"
groq_client = Groq(api_key=GROQ_API_KEY)

print("Generation:", GROQ_MODEL)
print("Judge:", JUDGE_PROVIDER, JUDGE_MODEL)

Generation: qwen/qwen3.6-27b
Judge: groq openai/gpt-oss-20b


In [25]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = "outputs/graphrag_eval_checkpoint.csv"
os.makedirs(os.path.dirname(CHECKPOINT) or ".", exist_ok=True)

def run_evaluation(golden_df):
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)


✅ Golden Dataset valid.


Evaluation:   0%|          | 0/5 [00:00<?, ?it/s][#E930]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-28944c2f-f40c-0001.production-orch-1163.neo4j.io', 7687)) (ResolvedIPv4Address(('35.220.238.105', 7687))): ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)


⚠️ Neo4j SessionExpired (lần 1/3). Reconnect sau 1.6s...


Evaluation:  60%|██████    | 3/5 [00:47<00:34, 17.29s/it][#C392]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-28944c2f-f40c-0001.production-orch-1163.neo4j.io', 7687)) (ResolvedIPv4Address(('35.220.238.105', 7687))): ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)


⚠️ Neo4j SessionExpired (lần 1/3). Reconnect sau 1.4s...


Evaluation:  80%|████████  | 4/5 [01:28<00:26, 26.39s/it][#CD55]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('35.220.238.105', 7687)) (ResolvedIPv4Address(('35.220.238.105', 7687))): ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)
Unable to retrieve routing information


⚠️ Neo4j ServiceUnavailable (lần 1/3). Reconnect sau 1.3s...


Evaluation: 100%|██████████| 5/5 [02:17<00:00, 27.45s/it]


,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G01,factoid,Who was the CEO of Hugging Face in 2023?,Clément Delangue,<think>\nHere's a thinking process:\n\n1. **Analyze User Input:**\n - **Question:** Who was the CEO of Hugging Fa...,<think>\nHere's a thinking process:\n\n1. **Analyze User Input:**\n - **Question:** Who was the CEO of Hugging Fa...,1,1,1,1,1,1,2.159301,2.242263,1703,1552,"The candidate answer states that the supplied context lacks information about Hugging Face’s CEO, whereas the refere...","The candidate answer states that the supplied context lacks information about Hugging Face’s CEO, whereas the refere...",0
1,G02,multi-hop,"Which company acquired the startup founded by the creator of technology X, and when?",Microsoft acquired Nuance Communications.,"<think>\nHere's a thinking process:\n\n1. **Analyze User Question:**\n - Question: ""Which company acquired the st...",<think>\nHere's a thinking process:\n\n1. **Analyze User Input:**\n - **Question:** Which company acquired the st...,1,1,1,1,1,1,2.242668,1.443371,1765,1133,The candidate answer does not provide the correct information (Microsoft acquired Nuance Communications) and incorre...,The candidate answer does not provide the correct information from the reference answer. It incorrectly states that ...,0
2,G03,cross-doc,Synthesize the partnership timeline between Company A and Company B across 2021-2023.,"OpenAI and Microsoft: 2019 initial investment, expanded in 2021 and 2023 multi-billion investment with Azure exclusi...",<think>\nHere's a thinking process:\n\n1. **Analyze User Input:**\n - **Question:** Synthesize the partnership ti...,<think>\nHere's a thinking process:\n\n1. **Analyze User Input:**\n - **Question:** Synthesize the partnership ti...,1,1,1,1,1,1,3.331459,15.039174,2329,1807,"The candidate answer incorrectly states that there is no evidence of a partnership between Company A and Company B, ...","The candidate answer incorrectly states that there is no information about the partnership, whereas the reference an...",0
3,G04,multi-hop,What AI product was developed by the organization that received funding from both Google and Amazon?,Anthropic received investment from Google/Amazon and developed Claude AI.,"<think>\nHere's a thinking process:\n\n1. **Analyze User Question:** ""What AI product was developed by the organiza...","<think>\nHere's a thinking process:\n\n1. **Analyze User Question:** ""What AI product was developed by the organiza...",1,1,1,1,1,1,13.016750,10.795096,2003,1021,"The candidate answer fails to provide the correct AI product, Claude AI, which is explicitly stated in the reference...","The candidate answer incorrectly states that the information is missing, whereas the reference answer provides a cle...",0
4,G05,cross-doc,Identify one technology connected to the same company in at least two news chunks and summarize how the relationship...,Microsoft and Azure OpenAI services: shifted from initial investment to full product integration in Office 365.,<think>\nHere's a thinking process:\n\n1. **Analyze User Input:**\n - **Question:** Identify one technology conne...,<think>\nThinking Process:\n1. **Analyze the Request:**\n * Task: Identify one technology connected to the sam...,1,1,1,1,1,1,8.311400,16.283109,2739,2374,"The candidate answer incorrectly states that there is insufficient evidence, whereas the reference answer provides a...","The candidate answer incorrectly claims insufficient evidence, whereas the reference answer provides a clear example...",0


In [26]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

os.makedirs("outputs", exist_ok=True)
comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv("outputs/graphrag_eval_results.csv", index=False)
comparison_df.to_csv("outputs/graphrag_vs_flatrag_summary.csv", index=False)
print("✅ Đã xuất báo cáo thành công vào outputs/graphrag_eval_results.csv và outputs/graphrag_vs_flatrag_summary.csv")


,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,1.000,1.000,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,1.000,1.000,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,1.000,1.000,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),5.821,15.661,Flat RAG thường rẻ/nhanh hơn.
4,cross-doc,Token usage,2534.000,2090.500,GraphRAG không đắt hơn trong sample này.
5,factoid,Comprehensiveness,1.000,1.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,1.000,1.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,1.000,1.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),2.159,2.242,Flat RAG thường rẻ/nhanh hơn.
9,factoid,Token usage,1703.000,1552.000,GraphRAG không đắt hơn trong sample này.


✅ Đã xuất báo cáo thành công vào outputs/graphrag_eval_results.csv và outputs/graphrag_vs_flatrag_summary.csv


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [29]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    rejected = audit_df[audit_df.decision.astype(str).str.startswith("REJECT")]
    if not rejected.empty:
        display(rejected.sort_values("similarity", ascending=False).head(20))
    else:
        print("Không có cặp nào bị reject.")

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)


{'id': '7b219357277193c25d37d788', 'name': 'Railergy', 'degree': 5} fetched= 5
No audit rows.


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [ ]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()

In [ ]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau